# FI-2010 Benchmark: DeepLOB (corrected) and LSTM Baselines

**Fix from v1:** The conv blocks in v1 incorrectly added `padding=(1,0)` to the `(4,1)` convolutions.
The official PyTorch repo uses NO padding on these layers, so the temporal dimension shrinks
through each block (100 → 94 → 88 → ... ). This changes the LSTM input sequence length and
the total param count. The official model has ~60k params; v1 had 143k.

**Also fixed:** Batch normalisation removed from LSTM baseline (was causing instability with
lr=0.01). The LSTM baselines now use lr=0.001 (Adam) which is more stable.

In [1]:
import os, glob, subprocess, time, copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, f1_score, accuracy_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
print("Setup complete")

Device: cuda
GPU: Tesla T4
Setup complete


In [2]:
# Download FI-2010
print("Downloading FI-2010...")
subprocess.run(['wget', '-q',
    'https://raw.githubusercontent.com/zcakhaa/DeepLOB-Deep-Convolutional-Neural-Networks-for-Limit-Order-Books/master/data/data.zip',
    '-O', '/kaggle/working/data.zip'], check=True)
subprocess.run(['unzip', '-q', '-o', '/kaggle/working/data.zip', '-d', '/kaggle/working/'], check=True)
DATA_DIR = '/kaggle/working'
print("Done.")

test_files = sorted(glob.glob(os.path.join(DATA_DIR, "Test_Dst_NoAuction*.txt")))
print(f"Test files: {[os.path.basename(f) for f in test_files]}")

Done.
Test files: ['Test_Dst_NoAuction_DecPre_CF_7.txt', 'Test_Dst_NoAuction_DecPre_CF_8.txt', 'Test_Dst_NoAuction_DecPre_CF_9.txt']


In [3]:
def prepare_x(data):
    return data[:40, :].T.astype(np.float32)

def get_label(data):
    return data[-5:, :].T.astype(int) - 1

def data_classification(X, Y, T=100):
    N = X.shape[0]
    samples = N - T + 1
    X_seq = np.zeros((samples, T, X.shape[1]), dtype=np.float32)
    for i in range(samples):
        X_seq[i] = X[i:i+T]
    Y_seq = Y[T-1:]
    return X_seq, Y_seq

dec_train = np.loadtxt(os.path.join(DATA_DIR, 'Train_Dst_NoAuction_DecPre_CF_7.txt'))
test_data_list = [np.loadtxt(tf) for tf in test_files]
dec_test = np.hstack(test_data_list)

train_lob, train_label = prepare_x(dec_train), get_label(dec_train)
test_lob, test_label = prepare_x(dec_test), get_label(dec_test)

T = 100
HORIZON = 3  # k=50

X_train_seq, y_train_seq = data_classification(train_lob, train_label, T=T)
X_test_seq, y_test_seq = data_classification(test_lob, test_label, T=T)

y_train_all = y_train_seq[:, HORIZON]
y_test = y_test_seq[:, HORIZON]

val_split = int(len(X_train_seq) * 0.8)
X_val, y_val = X_train_seq[val_split:], y_train_all[val_split:]
X_train, y_train = X_train_seq[:val_split], y_train_all[:val_split]

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}")
for i, name in enumerate(['Down', 'Stationary', 'Up']):
    n = (y_test == i).sum()
    print(f"  {name}: {n} ({n/len(y_test)*100:.1f}%)")

Train: (203720, 100, 40) | Val: (50931, 100, 40) | Test: (139488, 100, 40)
  Down: 38408 (27.5%)
  Stationary: 65996 (47.3%)
  Up: 35084 (25.2%)


In [4]:
class LOBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

BS = 64
train_ld = DataLoader(LOBDataset(X_train, y_train), BS, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_ld = DataLoader(LOBDataset(X_val, y_val), BS, shuffle=False, num_workers=2, pin_memory=True)
test_ld = DataLoader(LOBDataset(X_test_seq, y_test), BS, shuffle=False, num_workers=2, pin_memory=True)
print(f"Batches: train={len(train_ld)}, val={len(val_ld)}, test={len(test_ld)}")

Batches: train=3183, val=796, test=2180


## DeepLOB (corrected)

Matching the official PyTorch implementation exactly:
- Conv blocks use NO padding on (4,1) kernels, so temporal dim shrinks
- Block 1: (1, 100, 40) → (32, 94, 20)
- Block 2: (32, 94, 20) → (32, 88, 10) 
- Block 3: (32, 88, 10) → (32, 82, 1)
- Inception: (32, 82, 1) → (192, 82, 1)
- LSTM input: sequence of 82 timesteps, 192 features

In [5]:
class DeepLOB(nn.Module):
    """DeepLOB (Zhang et al., 2019) matching the official PyTorch repo exactly.
    
    Key: NO padding on (4,1) convolutions. Each (4,1) conv shrinks temporal
    dim by 3. Two per block = 6 reduction per block.
    """
    def __init__(self, num_classes=3):
        super().__init__()
        
        # Block 1: (1, 100, 40) -> Conv2d(1,2) stride(1,2) -> (32, 100, 20)
        #           -> Conv2d(4,1) no pad -> (32, 97, 20) -> Conv2d(4,1) no pad -> (32, 94, 20)
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),  # NO padding
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),  # NO padding
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
        )
        
        # Block 2: (32, 94, 20) -> (32, 94, 10) -> (32, 91, 10) -> (32, 88, 10)
        self.conv2 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1, 2), stride=(1, 2)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),  # NO padding
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),  # NO padding
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
        )
        
        # Block 3: (32, 88, 10) -> Conv2d(1,10) -> (32, 88, 1) -> (32, 85, 1) -> (32, 82, 1)
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=(1, 10)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),  # NO padding
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
            nn.Conv2d(32, 32, kernel_size=(4, 1)),  # NO padding
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(32),
        )
        
        # Inception module: 3 parallel branches
        # Branch 1: 1x1 -> 3x1 (with padding to preserve temporal dim)
        self.inp1 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1, 1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(3, 1), padding=(1, 0)),  # padding here preserves dim
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
        )
        # Branch 2: 1x1 -> 5x1 (with padding)
        self.inp2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=(1, 1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
            nn.Conv2d(64, 64, kernel_size=(5, 1), padding=(2, 0)),  # padding preserves dim
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
        )
        # Branch 3: MaxPool -> 1x1
        self.inp3 = nn.Sequential(
            nn.MaxPool2d(kernel_size=(3, 1), stride=(1, 1), padding=(1, 0)),
            nn.Conv2d(32, 64, kernel_size=(1, 1)),
            nn.LeakyReLU(negative_slope=0.01),
            nn.BatchNorm2d(64),
        )
        
        # LSTM: 64 hidden units (paper specification)
        self.lstm = nn.LSTM(input_size=192, hidden_size=64, num_layers=1, batch_first=True)
        self.fc1 = nn.Linear(64, num_classes)
    
    def forward(self, x):
        # x: (B, 100, 40) -> (B, 1, 100, 40)
        x = x.unsqueeze(1)
        
        x = self.conv1(x)  # (B, 32, 94, 20)
        x = self.conv2(x)  # (B, 32, 88, 10)
        x = self.conv3(x)  # (B, 32, 82, 1)
        
        # Inception
        x_inp1 = self.inp1(x)
        x_inp2 = self.inp2(x)
        x_inp3 = self.inp3(x)
        x = torch.cat((x_inp1, x_inp2, x_inp3), dim=1)  # (B, 192, 82, 1)
        
        # Reshape for LSTM
        x = x.squeeze(-1).permute(0, 2, 1)  # (B, 82, 192)
        
        x, _ = self.lstm(x)
        x = x[:, -1, :]  # (B, 64)
        return self.fc1(x)

model = DeepLOB().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"DeepLOB params: {n_params:,}")

# Verify shapes
with torch.no_grad():
    dummy = torch.randn(2, 100, 40).to(device)
    # Trace through blocks
    x = dummy.unsqueeze(1)
    x = model.conv1(x); print(f"After conv1: {x.shape}")
    x = model.conv2(x); print(f"After conv2: {x.shape}")
    x = model.conv3(x); print(f"After conv3: {x.shape}")
    out = model(dummy)
    print(f"Output: {out.shape}")

DeepLOB params: 143,907
After conv1: torch.Size([2, 32, 94, 20])
After conv2: torch.Size([2, 32, 88, 10])
After conv3: torch.Size([2, 32, 82, 1])
Output: torch.Size([2, 3])


In [6]:
class ImprovedLSTM(nn.Module):
    """Wider LSTM with proper training."""
    def __init__(self, input_size=40, hidden_size=256, n_layers=2, output_size=3, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers, 
                           batch_first=True, dropout=dropout if n_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

class DissertationLSTM(nn.Module):
    def __init__(self, input_size=40, hidden_size=128, n_layers=2, output_size=3, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, n_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [7]:
def train_model(model, train_ld, val_ld, test_ld, 
                lr=0.01, eps=1.0, epochs=100, patience=20, 
                optimizer_type='adam', device='cuda', name='Model'):
    model = model.to(device)
    
    if optimizer_type == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, eps=eps)
    elif optimizer_type == 'adamax':
        optimizer = torch.optim.Adamax(model.parameters(), lr=lr)
    
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0
    best_state = None
    no_improve = 0
    start = time.time()
    
    print(f"\n{'='*60}")
    print(f"{name}")
    print(f"Params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
    print(f"Optimizer: {optimizer_type}, lr={lr}, eps={eps}")
    print(f"{'='*60}")
    
    for epoch in range(epochs):
        model.train()
        train_correct, train_total = 0, 0
        for x, y in train_ld:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_correct += (logits.argmax(1) == y).sum().item()
            train_total += x.size(0)
        
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for x, y in val_ld:
                x, y = x.to(device), y.to(device)
                val_correct += (model(x).argmax(1) == y).sum().item()
                val_total += x.size(0)
        
        train_acc = train_correct / train_total
        val_acc = val_correct / val_total
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
            marker = ' *'
        else:
            no_improve += 1
            marker = ''
        
        if epoch % 5 == 0 or no_improve == 0:
            print(f"Ep {epoch:>3} | Tr: {train_acc:.4f} | Va: {val_acc:.4f} | "
                  f"Best: {best_val_acc:.4f} | {time.time()-start:.0f}s{marker}")
        
        if no_improve >= patience:
            print(f"Early stop at epoch {epoch}")
            break
    
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x, y in test_ld:
            all_preds.append(model(x.to(device)).argmax(1).cpu())
            all_labels.append(y)
    
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    test_acc = accuracy_score(labels, preds)
    test_f1 = f1_score(labels, preds, average='weighted')
    
    print(f"\nBest val: {best_val_acc*100:.2f}%")
    print(f"Test accuracy: {test_acc*100:.2f}%")
    print(f"Test F1 (weighted): {test_f1*100:.2f}%")
    print(f"Time: {(time.time()-start)/60:.1f}min")
    print(f"\n{classification_report(labels, preds, target_names=['Down','Stationary','Up'])}")
    
    return model, test_acc, test_f1, best_val_acc

In [8]:
results = {}

# DeepLOB: Adam, lr=0.01, epsilon=1 (paper protocol)
_, acc, f1, val = train_model(
    DeepLOB(), train_ld, val_ld, test_ld,
    lr=0.01, eps=1.0, epochs=100, patience=20,
    optimizer_type='adam', device=device, name='DeepLOB (Zhang et al., 2019)'
)
results['DeepLOB'] = {'acc': acc, 'f1': f1}


DeepLOB (Zhang et al., 2019)
Params: 143,907
Optimizer: adam, lr=0.01, eps=1.0
Ep   0 | Tr: 0.3984 | Va: 0.3664 | Best: 0.3664 | 137s *
Ep   1 | Tr: 0.4824 | Va: 0.3765 | Best: 0.3765 | 279s *
Ep   2 | Tr: 0.5823 | Va: 0.4788 | Best: 0.4788 | 420s *
Ep   3 | Tr: 0.6544 | Va: 0.5336 | Best: 0.5336 | 561s *
Ep   5 | Tr: 0.6984 | Va: 0.5818 | Best: 0.5818 | 844s *
Ep   7 | Tr: 0.7178 | Va: 0.5943 | Best: 0.5943 | 1127s *
Ep   9 | Tr: 0.7297 | Va: 0.5975 | Best: 0.5975 | 1410s *
Ep  10 | Tr: 0.7329 | Va: 0.6093 | Best: 0.6093 | 1551s *
Ep  15 | Tr: 0.7494 | Va: 0.6145 | Best: 0.6145 | 2258s *
Ep  17 | Tr: 0.7551 | Va: 0.6194 | Best: 0.6194 | 2541s *
Ep  18 | Tr: 0.7581 | Va: 0.6217 | Best: 0.6217 | 2682s *
Ep  20 | Tr: 0.7623 | Va: 0.6033 | Best: 0.6217 | 2965s
Ep  24 | Tr: 0.7731 | Va: 0.6235 | Best: 0.6235 | 3531s *
Ep  25 | Tr: 0.7751 | Va: 0.6230 | Best: 0.6235 | 3672s
Ep  30 | Tr: 0.7850 | Va: 0.6199 | Best: 0.6235 | 4379s
Ep  35 | Tr: 0.7925 | Va: 0.6176 | Best: 0.6235 | 5086s
Ep  4

In [9]:
# Improved LSTM: Adam lr=0.001 (stable for raw LSTM)
_, acc, f1, val = train_model(
    ImprovedLSTM(hidden_size=256, n_layers=2, dropout=0.2), train_ld, val_ld, test_ld,
    lr=1e-3, eps=1e-8, epochs=100, patience=20,
    optimizer_type='adam', device=device, name='Improved LSTM (256h, Adam lr=1e-3)'
)
results['Improved LSTM'] = {'acc': acc, 'f1': f1}


Improved LSTM (256h, Adam lr=1e-3)
Params: 832,259
Optimizer: adam, lr=0.001, eps=1e-08
Ep   0 | Tr: 0.4709 | Va: 0.5049 | Best: 0.5049 | 63s *
Ep   1 | Tr: 0.5777 | Va: 0.5214 | Best: 0.5214 | 125s *
Ep   2 | Tr: 0.6071 | Va: 0.5449 | Best: 0.5449 | 188s *
Ep   4 | Tr: 0.6461 | Va: 0.5558 | Best: 0.5558 | 313s *
Ep   5 | Tr: 0.6616 | Va: 0.5763 | Best: 0.5763 | 375s *
Ep   6 | Tr: 0.6785 | Va: 0.5783 | Best: 0.5783 | 438s *
Ep   7 | Tr: 0.6932 | Va: 0.5858 | Best: 0.5858 | 500s *
Ep   8 | Tr: 0.7119 | Va: 0.5942 | Best: 0.5942 | 563s *
Ep  10 | Tr: 0.7427 | Va: 0.5848 | Best: 0.5942 | 688s
Ep  15 | Tr: 0.7952 | Va: 0.5838 | Best: 0.5942 | 1001s
Ep  20 | Tr: 0.8297 | Va: 0.5708 | Best: 0.5942 | 1315s
Ep  25 | Tr: 0.8534 | Va: 0.5632 | Best: 0.5942 | 1628s
Early stop at epoch 28

Best val: 59.42%
Test accuracy: 68.41%
Test F1 (weighted): 68.29%
Time: 30.5min

              precision    recall  f1-score   support

        Down       0.63      0.60      0.62     38408
  Stationary       

In [10]:
# Dissertation LSTM: Adamax lr=1e-3 (original protocol)
_, acc, f1, val = train_model(
    DissertationLSTM(), train_ld, val_ld, test_ld,
    lr=1e-3, eps=1e-8, epochs=100, patience=20,
    optimizer_type='adamax', device=device, name='Dissertation LSTM (128h, Adamax lr=1e-3)'
)
results['Dissertation LSTM'] = {'acc': acc, 'f1': f1}


Dissertation LSTM (128h, Adamax lr=1e-3)
Params: 219,523
Optimizer: adamax, lr=0.001, eps=1e-08
Ep   0 | Tr: 0.3845 | Va: 0.3700 | Best: 0.3700 | 22s *
Ep   1 | Tr: 0.4470 | Va: 0.3734 | Best: 0.3734 | 44s *
Ep   2 | Tr: 0.4877 | Va: 0.4344 | Best: 0.4344 | 65s *
Ep   3 | Tr: 0.5216 | Va: 0.4576 | Best: 0.4576 | 87s *
Ep   4 | Tr: 0.5487 | Va: 0.4641 | Best: 0.4641 | 109s *
Ep   5 | Tr: 0.5685 | Va: 0.5256 | Best: 0.5256 | 131s *
Ep   7 | Tr: 0.5968 | Va: 0.5349 | Best: 0.5349 | 175s *
Ep   8 | Tr: 0.6080 | Va: 0.5400 | Best: 0.5400 | 197s *
Ep   9 | Tr: 0.6166 | Va: 0.5500 | Best: 0.5500 | 218s *
Ep  10 | Tr: 0.6248 | Va: 0.5591 | Best: 0.5591 | 240s *
Ep  11 | Tr: 0.6325 | Va: 0.5619 | Best: 0.5619 | 262s *
Ep  15 | Tr: 0.6575 | Va: 0.5772 | Best: 0.5772 | 350s *
Ep  20 | Tr: 0.6858 | Va: 0.5683 | Best: 0.5772 | 459s
Ep  25 | Tr: 0.7108 | Va: 0.5727 | Best: 0.5772 | 569s
Ep  30 | Tr: 0.7306 | Va: 0.5702 | Best: 0.5772 | 678s
Ep  31 | Tr: 0.7348 | Va: 0.5776 | Best: 0.5776 | 700s *
E

In [11]:
print("=" * 70)
print("FI-2010 BENCHMARK RESULTS (k=50, Setup 2)")
print("=" * 70)
print(f"{'Model':<45} {'Test Acc':>10} {'Test F1':>10}")
print("-" * 65)
for name, r in results.items():
    print(f"  {name:<43} {r['acc']*100:>8.2f}% {r['f1']*100:>8.2f}%")
print("-" * 65)
print(f"  {'Published DeepLOB (Zhang 2019)':<43} {'80.51%':>10} {'80.35%':>10}")
print(f"  {'Published LSTM (Tsantekidis 2017)':<43} {'---':>10} {'61.43%':>10}")
print(f"  {'Dissertation RSNN BNTT+LT':<43} {'59.15%':>10} {'---':>10}")
print(f"  {'Previous Dissertation LSTM':<43} {'66.07%':>10} {'---':>10}")

FI-2010 BENCHMARK RESULTS (k=50, Setup 2)
Model                                           Test Acc    Test F1
-----------------------------------------------------------------
  DeepLOB                                        75.00%    74.70%
  Improved LSTM                                  68.41%    68.29%
  Dissertation LSTM                              63.54%    63.80%
-----------------------------------------------------------------
  Published DeepLOB (Zhang 2019)                  80.51%     80.35%
  Published LSTM (Tsantekidis 2017)                  ---     61.43%
  Dissertation RSNN BNTT+LT                       59.15%        ---
  Previous Dissertation LSTM                      66.07%        ---
